# register-buffer — worked example 1: Register a Running Mean Buffer in a Custom Module

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `register-buffer`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In PyTorch, `self.register_buffer(name, tensor)` attaches a tensor to a Module such that it appears in `state_dict()` (and is saved/loaded with checkpoints) but is NOT included in `.parameters()` (so it is ignored by optimizers). This is the right choice for statistics like running means that are updated manually during training, not through gradient descent.

## Worked solution

**Step 1 — subclass `nn.Module`.** Any custom module extends `nn.Module` and calls `super().__init__()` first.

**Step 2 — register the learnable weight as `nn.Parameter`.** Wrap a tensor in `nn.Parameter(...)` to make it trainable. It will appear in both `.parameters()` and `state_dict()`.

**Step 3 — register the running mean as a buffer.** Call `self.register_buffer('running_mean', torch.zeros(dim))`. The name `'running_mean'` becomes an attribute (accessible as `self.running_mean`) and appears in `state_dict()` but not `.parameters()`.

**Step 4 — verify the discrimination.** After instantiation, iterate `.parameters()` and `.buffers()`. `weight` appears only in `.parameters()`, `running_mean` appears only in `.buffers()`. Both appear in `state_dict().keys()`.

**Step 5 — check `requires_grad`.** Buffers have `requires_grad=False` by default; parameters have `requires_grad=True`.

In [ ]:
import torch as t
import torch.nn as nn

class EMATracker(nn.Module):
    """Tracks an exponential moving average (running mean) of input statistics."""
    def __init__(self, dim: int, momentum: float = 0.1):
        super().__init__()
        # Trainable scale applied after normalization
        self.scale = nn.Parameter(t.ones(dim))
        # Running mean: updated manually, not by optimizer
        self.register_buffer('running_mean', t.zeros(dim))
        self.momentum = momentum

    def forward(self, x: t.Tensor) -> t.Tensor:
        # Update running mean (during training in real use)
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * x.detach()
        return (x - self.running_mean) * self.scale

# --- exercise and print ---
module = EMATracker(dim=4)

param_names = [n for n, _ in module.named_parameters()]
buffer_names = [n for n, _ in module.named_buffers()]
sd_keys = list(module.state_dict().keys())

print('Parameters:', param_names)     # ['scale']
print('Buffers:   ', buffer_names)    # ['running_mean']
print('State dict:', sd_keys)         # ['scale', 'running_mean']
print('scale.requires_grad:', module.scale.requires_grad)                 # True
print('running_mean.requires_grad:', module.running_mean.requires_grad)   # False